# Install dependencies

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv('var.env')
hf_token = os.getenv("HF_TOKEN")

In [ ]:
!pip -q install transformers datasets peft accelerate bitsandbytes trl

# Create training dataset

In [ ]:
from datasets import Dataset

data = [
    {
        "instruction": "What is CAN bus?",
        "output": "CAN bus is a vehicle communication protocol..."
    },
    {
        "instruction": "Explain telematics anomaly detection",
        "output": "Telematics anomaly detection identifies unusual vehicle patterns..."
    },
    {
        "instruction": "What is VIN?",
        "output": "Vehicle Identification Number is a unique identifier..."
    }
]

dataset = Dataset.from_list(data)

# Convert dataset to chat format

In [ ]:
def format_dataset(record):
  return {
      "text": f"### Instruction:\n{record['instruction']}\n\n### Response:\n{record['output']}"
  }

dataset = dataset.map(format_dataset)

# Load model with quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 1. Define the quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16", # Common for T4 GPUs
    bnb_4bit_quant_type="nf4"         # Recommended for better precision
)

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map='auto',
    token=hf_token
)


# Finetune the model - Applying LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

Only 0.09% of trainable parameters

# Tokenize dataset

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize(example):
    outputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    # The Trainer looks for this 'labels' key to calculate loss
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

dataset = dataset.map(tokenize)

# Training

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    output_dir="finetuned_model"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

# Save finetuned model

In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Inference

In [ ]:
prompt = "what is VIN"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0]))